# Overton: keyword mode vs semantic mode

Overton's `/documents.php` endpoint answers two different kinds of question,
and which one you use changes the results completely.

| mode | parameter | matches against | good for |
|---|---|---|---|
| **keyword** | `query=` | the document's own words and title | finding a *named* document, boolean/field-prefixed searching |
| **semantic** | `squery=` (+ `min_similarity`) | an AI-written description of the document | finding documents *about* a topic, from intent text |

**Where each is used today.** The Policy Atlas pipeline searches Overton in
semantic mode only — that is its v1 design decision (see
`docs/tasks/015-live-search/api-filter-research.md`). The ground-truth builder
in `ground_truth.py` uses keyword mode, because it is looking up one exact
cited document rather than exploring a topic.

That difference is worth probing. If Overton recall in the eval looks poor,
the first hypothesis to test is that semantic-only retrieval cannot find
documents the reference list names precisely.

## How to launch

From the repo root, so the environment file and the package are both found:

```
uv run --project backend --env-file backend/.env jupyter lab
```

Then open this notebook. If you launch Jupyter another way, the next cell
loads `backend/.env` itself, so `OVERTON_API_KEY` is available either way.

**Rate limit:** every call below waits 1.2 seconds afterwards, which is
Overton's pacing floor. A cell that makes six calls takes about seven seconds.

In [ ]:
import difflib
import html
import os
import time
from typing import Any

import httpx
import pandas as pd
from dotenv import load_dotenv

# The notebook's own directory is the working directory, so backend/.env is
# two levels up. Harmless if the key is already in the environment.
load_dotenv("../../backend/.env")

OVERTON_HOST = "https://app.overton.io"
MIN_INTERVAL_S = 1.2  # Overton's pacing floor
API_KEY = os.environ.get("OVERTON_API_KEY")
print("OVERTON_API_KEY:", "found" if API_KEY else "MISSING — check backend/.env")

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)

In [ ]:
def overton_call(params: dict[str, str], n: int = 5) -> list[dict[str, Any]]:
    """One raw call to Overton, paced. Returns the results list.

    Deliberately raw rather than going through the pipeline's backend class:
    that class fixes `squery` and `min_similarity`, which are the two things
    worth varying here.
    """
    resp = httpx.get(
        f"{OVERTON_HOST}/documents.php",
        params={**params, "format": "json", "api_key": API_KEY, "pp": str(n)},
        timeout=30.0,
    )
    time.sleep(MIN_INTERVAL_S)
    resp.raise_for_status()
    body = resp.json()
    return body.get("results", []) or []


def title_of(record: dict[str, Any]) -> str:
    """Display title, with HTML entities decoded — Overton ships `&#39;` raw."""
    return html.unescape(record.get("translated_title") or record.get("title") or "")


def match_form(text: str) -> str:
    """Lowercased, curly quotes folded to straight — for comparing titles."""
    return text.lower().replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')


def frame(records: list[dict[str, Any]], mode: str, against: str | None = None) -> pd.DataFrame:
    """Results as a table. `against` adds a similarity column, so you can see
    at a glance whether a mode actually returned the document you asked for.
    """
    rows = []
    for rank, record in enumerate(records, start=1):
        source = record.get("source") or {}
        title = title_of(record)
        row = {
            "mode": mode,
            "rank": rank,
            "title": title,
            "publisher": source.get("title"),
            "country": source.get("country"),
            "published": record.get("published_on"),
            "id": record.get("policy_document_id"),
        }
        if against:
            row["similarity"] = round(
                difflib.SequenceMatcher(None, match_form(title), match_form(against)).ratio(), 2
            )
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def keyword(query: str, n: int = 5, **filters: str) -> pd.DataFrame:
    """Keyword mode. Supports Overton's advanced syntax — see the last cell."""
    return frame(overton_call({"query": query, **filters}, n), "keyword")


def semantic(text: str, n: int = 5, min_similarity: float = 0.3, **filters: str) -> pd.DataFrame:
    """Semantic mode — what the pipeline uses. `min_similarity` 0.3 is the
    pipeline's own setting; raise it to demand closer matches.
    """
    params = {"squery": text, "min_similarity": str(min_similarity), **filters}
    return frame(overton_call(params, n), f"semantic@{min_similarity}")


def compare(text: str, n: int = 5, min_similarity: float = 0.3) -> pd.DataFrame:
    """Both modes on the same input, stacked, scored against the input text.

    Three queries: the `title:"..."` field-prefixed form, the bare keyword
    form, and semantic. Colons are stripped from the prefixed form because a
    colon is field-prefix syntax and would match nothing.
    """
    cleaned = text.replace('"', " ").replace(":", " ").strip()
    parts = [
        frame(overton_call({"query": f'title:"{cleaned}"'}, n), 'keyword title:"..."', against=text),
        frame(overton_call({"query": cleaned}, n), "keyword bare", against=text),
        frame(
            overton_call({"squery": text, "min_similarity": str(min_similarity)}, n),
            "semantic",
            against=text,
        ),
    ]
    return pd.concat(parts, ignore_index=True)

## 1. Looking for one named document

An ONS statistical bulletin cited by the GOV.UK loneliness evidence review.
Overton holds it. Watch which mode returns it.

In [ ]:
TITLE = "Loneliness - What characteristics and circumstances are associated with feeling lonely?"
compare(TITLE)

Expect keyword mode to return the document at a similarity near 1.0, and
semantic mode to return topically-related documents — often foreign-language
policy papers about loneliness — and not the document itself. Semantic mode is
not failing; it is answering a different question.

## 2. The apostrophe trap

Overton's keyword index treats the typographic apostrophe (’) as a different
character from the straight one ('). The same title finds nothing in one
spelling and the exact document in the other. `ground_truth.py` tries both.

In [ ]:
APOSTROPHE_TITLE = "Children's and young people's experiences of loneliness  2018"
pd.concat(
    [
        keyword(f'title:"{APOSTROPHE_TITLE}"').assign(spelling="straight '"),
        keyword(f'title:"{APOSTROPHE_TITLE.replace(chr(39), chr(8217))}"').assign(spelling="curly \u2019"),
    ],
    ignore_index=True,
)

## 3. Searching for a topic

This is what the pipeline actually does: it sends the research intent, or an
LLM paraphrase of it, as `squery`. Semantic mode wants descriptive prose —
Overton's own guidance is two or more sentences. Compare a bare keyword
phrase against a full sentence of intent.

In [ ]:
INTENT = (
    "What interventions reduce loneliness among older adults, and what does the "
    "evidence say about their effectiveness? Evaluations of befriending schemes, "
    "social prescribing and community programmes are of particular interest."
)
semantic(INTENT, n=10)

In [ ]:
compare("The cost of loneliness to UK employers")

In [ ]:
# Same intent, keyword mode. Keyword mode has no idea what to do with a
# sentence — every word is a term to match.
keyword(INTENT, n=10)

In [ ]:
# Raising min_similarity demands closer semantic matches. Too high and it
# returns nothing at all — worth knowing where that cliff is.
pd.concat([semantic(INTENT, n=5, min_similarity=s) for s in (0.3, 0.5, 0.7)], ignore_index=True)

## 4. Does the eval's resolver find a given citation?

The exact function the ground-truth builder uses, so you can check a citation
before wondering why it is missing from a report's `overton_ids`.

In [ ]:
from ground_truth import ExtractedCitation, resolve_citation_overton

CITATIONS = [
    "A connected society: a strategy for tackling loneliness",
    "Children's and young people's experiences of loneliness: 2018",
    "Community Life Survey 2017-18",
    "The role of transport in tackling loneliness",
]
pd.DataFrame(
    [
        {
            "title": title,
            "resolved_key": resolve_citation_overton(
                ExtractedCitation(raw_citation=title, title_guess=title)
            ),
        }
        for title in CITATIONS
    ]
)

A `None` means Overton has no document whose title matches closely enough
(the floor is `ground_truth._TITLE_MATCH_THRESHOLD`, 0.82). Use `compare()`
above on that title to see whether the document is there under a different
name, or genuinely absent — the two have different implications for the eval.

## 5. Your own queries

Edit and run.

In [ ]:
compare("Tackling loneliness annual report")

In [ ]:
# Filters work alongside either mode. These names come from the 015 research
# doc and are inferred rather than officially documented, so treat a zero
# result as "maybe the wrong token" before concluding "nothing exists".
keyword('title:"loneliness"', n=10, source_country="GB", published_after="2018-01-01")

## Keyword syntax cheat sheet

From `docs/tasks/015-live-search/api-filter-research.md`:

| syntax | meaning |
|---|---|
| `AND` `OR` `NOT`, parentheses | boolean logic |
| `"exact phrase"` | phrase match |
| `title:` `abstract:` `full-text:` `domain:` | field prefixes |
| `word1 word2~5` | proximity within 5 words |

Traps worth remembering:

- **Diacritics and apostrophes are significant** — see section 2.
- **A colon inside a phrase reads as a field prefix.** Strip it, as `compare()` does.
- **`min_similarity` applies to semantic mode only.**
- **Filter vocabularies are open**: a wrong token silently returns zero rows
  rather than an error, which looks exactly like an empty index.
- **`squery` + filter behaviour is undocumented.** Assume filters are applied
  after the semantic match, but verify before relying on it.